### 인구수 맵핑

In [1]:
import pandas as pd
import numpy as np

In [2]:
market_df_2 = pd.read_csv(r"C:\Users\legen\Desktop\Lab Project\BC\data\market_df_2.csv", encoding="utf-8-sig")

pop_df = pd.read_csv(r"D:\PP\BC\data\og\지역별(행정동) 성별 연령별 주민등록 인구수_20231031.csv", encoding="cp949")

In [3]:
market_df_2.columns

Index(['시장명', '지번주소', '도로명주소', '시도', '시군구', '아케이드 보유 여부', '엘리베이터_에스컬레이터_보유여부',
       '고객지원센터 보유 여부', '스프링쿨러 보유 여부', '화재감지기 보유여부', '유아놀이방_보유여부', '종합콜센터_보유여부',
       '고객휴게실_보유여부', '수유센터_보유여부', '물품보관함_보유여부', '자전거보관함_보유여부', '체육시설_보유여부',
       '간이 도서관_보유여부', '쇼핑카트_보유여부', '외국인 안내센터_보유여부', '고객동선통로_보유여부', '방송센터_보유여부',
       '문화교실_보유여부', '공동물류창고_보유여부', '시장전용 고객주차장_보유여부', '교육장_보유여부', '회의실_보유여부',
       '자동심장충격기_보유여부', '시장면적', '전체점포', '빈점포', '기타점포', '노점수', '점포상인', '종업원',
       '노점상인', '총시장상인', '지원금액', '지원여부', 'has_assoc', 'join_stores', 'delivery',
       'grocery', 'item_diversity', 'is_food_based', 'has_nonfood',
       'fresh_count', 'nonfood_count', 'market_item_type', 'dong_guess',
       'dong_guess_je', '행정기관코드', 'match_status'],
      dtype='object')

In [4]:
market_df_2 = market_df_2.dropna(subset = ['행정기관코드'])

In [5]:
# 행정기관코드 타입 맞추기
market_df_2["행정기관코드"] = market_df_2["행정기관코드"].round().astype("int64").astype(str)
pop_df["행정기관코드"] = pop_df["행정기관코드"].astype(str)


def sum_age_band(df, ages):
    cols_m = [f"만{a}세남자" for a in ages if f"만{a}세남자" in df.columns]
    cols_f = [f"만{a}세여자" for a in ages if f"만{a}세여자" in df.columns]
    return df[cols_m].sum(axis=1) + df[cols_f].sum(axis=1)

# pop 변수
pop = pop_df[["행정기관코드", "계", "남자", "여자"]].copy()
pop = pop.rename(columns={"계": "pop_t", "남자": "pop_m", "여자": "pop_f"})

pop["pop_0"]  = sum_age_band(pop_df, range(0, 10)).astype(int)   # 0~9
pop["pop_10"] = sum_age_band(pop_df, range(10, 20)).astype(int)  # 10~19
pop["pop_20"] = sum_age_band(pop_df, range(20, 30)).astype(int)  # 20~29
pop["pop_30"] = sum_age_band(pop_df, range(30, 40)).astype(int)  # 30~39
pop["pop_40"] = sum_age_band(pop_df, range(40, 50)).astype(int)  # 40~49
pop["pop_50"] = sum_age_band(pop_df, range(50, 60)).astype(int)  # 50~59

pop["pop_60"] = (sum_age_band(pop_df, range(60, 110))
                 + pop_df["만110세이상남자"]
                 + pop_df["만110세이상여자"]
                 ).astype(int) # 60 이상


In [6]:
# check
age_sum = pop[["pop_0","pop_10","pop_20","pop_30","pop_40","pop_50","pop_60"]].sum(axis=1)
(pop["pop_t"] - age_sum).abs().max()

np.int64(0)

In [7]:
market_df_2_merged = market_df_2.merge(
    pop[["행정기관코드","pop_t","pop_m","pop_f","pop_0","pop_10","pop_20","pop_30","pop_40","pop_50","pop_60"]],
    on="행정기관코드",
    how="left"
)

# 연령별 변수 생성
market_df_2_merged['pop_adole'] = market_df_2_merged["pop_10"]
market_df_2_merged["pop_young"]  = market_df_2_merged[["pop_20", "pop_30"]].sum(axis=1)
market_df_2_merged["pop_middle"] = market_df_2_merged[["pop_40", "pop_50"]].sum(axis=1)
market_df_2_merged["pop_senior"] = market_df_2_merged["pop_60"]
market_df_2_merged = market_df_2_merged.drop(columns= ['pop_0', 'pop_10', 'pop_20', 'pop_30', 'pop_40', 'pop_50', 'pop_60'])

In [8]:
market_df_2_merged.head(4)

,시장명,지번주소,도로명주소,시도,시군구,아케이드 보유 여부,엘리베이터_에스컬레이터_보유여부,고객지원센터 보유 여부,스프링쿨러 보유 여부,화재감지기 보유여부,...,dong_guess_je,행정기관코드,match_status,pop_t,pop_m,pop_f,pop_adole,pop_young,pop_middle,pop_senior
0,(주)신정시장,울산광역시 남구 신정동 630-1,울산광역시 남구 월평로47번길 7,울산광역시,남구,Y,N,Y,N,Y,...,NaN,3114051000,no_match,18163,9055,9108,1035,4575,5459,6306
1,수암상가시장,울산광역시 남구 야음동 697-9,울산광역시 남구 수암로 128번길 12,울산광역시,남구,Y,N,Y,Y,Y,...,NaN,3114062500,no_match,32595,16298,16297,3595,8231,10959,6326
2,수암종합시장,울산광역시 남구 야음동 698-1,울산광역시 남구 수암로 124 야음동,울산광역시,남구,Y,N,Y,N,Y,...,NaN,3114062500,no_match,32595,16298,16297,3595,8231,10959,6326
3,신정상가시장,울산광역시 남구 신정동 634-10,울산광역시 남구 중앙로241번길 14-1,울산광역시,남구,Y,N,Y,N,Y,...,NaN,3114051000,no_match,18163,9055,9108,1035,4575,5459,6306


In [9]:
market_df_2_merged.to_csv(r"C:\Users\legen\Desktop\Lab Project\BC\data\market_df_3.csv", encoding='utf-8-sig')